1.Write the python program to solve 8-puzzle problem

In [ ]:
import heapq

class PuzzleState:
    def __init__(self, board, g=0, parent=None):
        self.board = board
        self.g = g  # Cost from start to current state
        self.parent = parent
        self.h = self.manhattan_distance()  # Heuristic cost to goal
        self.f = self.g + self.h  # Total estimated cost

    def __lt__(self, other):
        return self.f < other.f

    def find_blank(self):
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == 0:
                    return i, j

    def manhattan_distance(self):
        distance = 0
        goal_positions = {
            1: (0, 0), 2: (0, 1), 3: (0, 2),
            4: (1, 0), 5: (1, 1), 6: (1, 2),
            7: (2, 0), 8: (2, 1), 0: (2, 2)
        }
        for i in range(3):
            for j in range(3):
                val = self.board[i][j]
                if val != 0:
                    target_i, target_j = goal_positions[val]
                    distance += abs(i - target_i) + abs(j - target_j)
        return distance

    def get_neighbors(self):
        neighbors = []
        r, c = self.find_blank()
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right

        for dr, dc in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                # Create a new board configuration
                new_board = [row[:] for row in self.board]
                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
                neighbors.append(PuzzleState(new_board, self.g + 1, self))
        return neighbors

    def is_goal(self):
        goal = [
            [1, 2, 3],
            [4, 5, 6],
            [7, 8, 0]
        ]
        return self.board == goal

def solve_8_puzzle(initial_board):
    initial_state = PuzzleState(initial_board)
    open_set = []
    heapq.heappush(open_set, initial_state)
    closed_set = set()

    while open_set:
        current_state = heapq.heappop(open_set)

        if current_state.is_goal():
            # Reconstruct the path
            path = []
            while current_state:
                path.append(current_state.board)
                current_state = current_state.parent
            return path[::-1]

        # Convert board to tuple to make it hashable for the closed set
        board_tuple = tuple(tuple(row) for row in current_state.board)
        closed_set.add(board_tuple)

        for neighbor in current_state.get_neighbors():
            neighbor_tuple = tuple(tuple(row) for row in neighbor.board)
            if neighbor_tuple in closed_set:
                continue

            heapq.heappush(open_set, neighbor)

    return None

# Helper to print the board nicely
def print_board(board):
    for row in board:
        print(" ".join(map(str, row)))
    print()

# Example usage:
# 0 represents the empty space
initial_configuration = [
    [1, 2, 3],
    [0, 4, 6],
    [7, 5, 8]
]

print("Solving 8-Puzzle from initial state:")
print_board(initial_configuration)

solution_path = solve_8_puzzle(initial_configuration)

if solution_path:
    print(f"Solved in {len(solution_path) - 1} moves!")
    for step, board in enumerate(solution_path):
        print(f"Step {step}:")
        print_board(board)
else:
    print("No solution found.")

Solving 8-Puzzle from initial state:
1 2 3
0 4 6
7 5 8

Solved in 3 moves!
Step 0:
1 2 3
0 4 6
7 5 8

Step 1:
1 2 3
4 0 6
7 5 8

Step 2:
1 2 3
4 5 6
7 0 8

Step 3:
1 2 3
4 5 6
7 8 0



2.Write the python program to solve 8-Queen problem

In [ ]:
def is_safe(board, row, col):
    # Check this row on left side
    for i in range(col):
        if board[row][i] == 1:
            return False

    # Check upper diagonal on left side
    for i, j in zip(range(row, -1, -1), range(col, -1, -1)):
        if board[i][j] == 1:
            return False

    # Check lower diagonal on left side
    for i, j in zip(range(row, 8, 1), range(col, -1, -1)):
        if board[i][j] == 1:
            return False

    return True

def solve_n_queens_util(board, col):
    # Base case: If all queens are placed, return True
    if col >= 8:
        return True

    # Consider this column and try placing this queen in all rows one by one
    for i in range(8):
        if is_safe(board, i, col):
            # Place this queen
            board[i][col] = 1

            # Recur to place rest of the queens
            if solve_n_queens_util(board, col + 1):
                return True

            # If placing queen in board[i][col] doesn't lead to a solution,
            # then remove queen from board[i][col]
            board[i][col] = 0

    # If the queen cannot be placed in any row in this column, return False
    return False

def solve_8_queens():
    board = [[0 for _ in range(8)] for _ in range(8)]

    if not solve_n_queens_util(board, 0):
        print("Solution does not exist")
        return False

    print_solution(board)
    return True

def print_solution(board):
    for i in range(8):
        row_str = []
        for j in range(8):
            if board[i][j] == 1:
                row_str.append("Q")
            else:
                row_str.append(".")
        print(" ".join(row_str))

# Solve and print
solve_8_queens()

Q . . . . . . .
. . . . . . Q .
. . . . Q . . .
. . . . . . . Q
. Q . . . . . .
. . . Q . . . .
. . . . . Q . .
. . Q . . . . .


True

3.Write the python program for Water Jug Problem

In [ ]:
from collections import deque

def solve_water_jug(jug1_cap, jug2_cap, target):
    # Keep track of visited states to avoid infinite loops
    # State representation: (amount_in_jug1, amount_in_jug2)
    visited = set()

    # Queue for BFS, storing path of states: [(jug1, jug2), ...]
    queue = deque([[(0, 0)]])

    while queue:
        path = queue.popleft()
        u, v = path[-1]  # Get the current state (last state in the path)

        # If target is achieved in either of the jugs, return the path
        if u == target or v == target:
            return path

        if (u, v) in visited:
            continue
        visited.add((u, v))

        # Generate all possible next states
        next_states = [
            (jug1_cap, v),      # Fill Jug 1
            (u, jug2_cap),      # Fill Jug 2
            (0, v),             # Empty Jug 1
            (u, 0),             # Empty Jug 2
            # Pour from Jug 2 to Jug 1
            (min(u + v, jug1_cap), v - (min(u + v, jug1_cap) - u)),
            # Pour from Jug 1 to Jug 2
            (u - (min(u + v, jug2_cap) - v), min(u + v, jug2_cap))
        ]

        for state in next_states:
            if state not in visited:
                new_path = list(path)
                new_path.append(state)
                queue.append(new_path)

    return None

# Define capacities and target
jug1_capacity = 4
jug2_capacity = 3
target_amt = 2

print(f"Solving Water Jug problem for capacities ({jug1_capacity}L, {jug2_capacity}L) and target {target_amt}L:\n")
solution = solve_water_jug(jug1_capacity, jug2_capacity, target_amt)

if solution:
    for step, (j1, j2) in enumerate(solution):
        print(f"Step {step}: Jug 1 = {j1}L, Jug 2 = {j2}L")
else:
    print("No solution exists for the given capacities and target.")

Solving Water Jug problem for capacities (4L, 3L) and target 2L:

Step 0: Jug 1 = 0L, Jug 2 = 0L
Step 1: Jug 1 = 0L, Jug 2 = 3L
Step 2: Jug 1 = 3L, Jug 2 = 0L
Step 3: Jug 1 = 3L, Jug 2 = 3L
Step 4: Jug 1 = 4L, Jug 2 = 2L


4.Write the python program for Cript-Arithmetic problem

In [ ]:
from itertools import permutations

def solve_cryptarithmetic():
    # Problem: SEND + MORE = MONEY
    # Letters involved: S, E, N, D, M, O, R, Y (8 unique letters)
    words = ["SEND", "MORE"]
    result = "MONEY"

    # Extract unique letters
    unique_letters = set("".join(words) + result)
    assert len(unique_letters) <= 10, "More than 10 unique letters! Not solvable in base 10."

    letters = list(unique_letters)
    first_letters = {w[0] for w in words + [result]}

    # Generate permutations of digits (0-9) matching the number of unique letters
    for p in permutations(range(10), len(letters)):
        char_to_digit = dict(zip(letters, p))

        # First letters of any word cannot be zero
        if any(char_to_digit[char] == 0 for char in first_letters):
            continue

        # Helper to convert word to its numerical value based on mapping
        def get_value(word):
            return sum(char_to_digit[char] * (10 ** i) for i, char in enumerate(reversed(word)))

        val_words = sum(get_value(word) for word in words)
        val_result = get_value(result)

        if val_words == val_result:
            print("Solution Found!")
            for char, digit in sorted(char_to_digit.items()):
                print(f"{char}: {digit}")
            print(f"\n{get_value('SEND')} + {get_value('MORE')} = {val_result}")
            return char_to_digit

    print("No solution exists.")
    return None

solve_cryptarithmetic()

Solution Found!
D: 7
E: 5
M: 1
N: 6
O: 0
R: 8
S: 9
Y: 2

9567 + 1085 = 10652


{'O': 0, 'S': 9, 'D': 7, 'E': 5, 'M': 1, 'R': 8, 'Y': 2, 'N': 6}

5.Write the python program for Missionaries Cannibal problem

In [ ]:
from collections import deque

class State:
    def __init__(self, m_left, c_left, boat, m_right, c_right):
        self.m_left = m_left
        self.c_left = c_left
        self.boat = boat  # 1 for left bank, 0 for right bank
        self.m_right = m_right
        self.c_right = c_right

    def is_valid(self):
        # Check for negative values or count exceeding initial total (3)
        if self.m_left < 0 or self.c_left < 0 or self.m_right < 0 or self.c_right < 0:
            return False
        # More cannibals than missionaries on the left bank
        if self.m_left > 0 and self.c_left > self.m_left:
            return False
        # More cannibals than missionaries on the right bank
        if self.m_right > 0 and self.c_right > self.m_right:
            return False
        return True

    def is_goal(self):
        return self.m_left == 0 and self.c_left == 0

    def __eq__(self, other):
        return (
            self.m_left == other.m_left and
            self.c_left == other.c_left and
            self.boat == other.boat and
            self.m_right == other.m_right and
            self.c_right == other.c_right
        )

    def __hash__(self):
        return hash((self.m_left, self.c_left, self.boat, self.m_right, self.c_right))

    def __repr__(self):
        boat_pos = "Left" if self.boat == 1 else "Right"
        return f"(M_left: {self.m_left}, C_left: {self.c_left}, Boat: {boat_pos}, M_right: {self.m_right}, C_right: {self.c_right})"

def get_successors(state):
    successors = []
    # All possible moves for 1 or 2 people crossing (M, C)
    moves = [(1, 0), (2, 0), (0, 1), (0, 2), (1, 1)]

    if state.boat == 1:  # Boat on left bank
        for dm, dc in moves:
            new_state = State(
                state.m_left - dm,
                state.c_left - dc,
                0,
                state.m_right + dm,
                state.c_right + dc
            )
            if new_state.is_valid():
                successors.append(new_state)
    else:  # Boat on right bank
        for dm, dc in moves:
            new_state = State(
                state.m_left + dm,
                state.c_left + dc,
                1,
                state.m_right - dm,
                state.c_right - dc
            )
            if new_state.is_valid():
                successors.append(new_state)
    return successors

def solve_missionaries_cannibals():
    # Initial state: 3 missionaries, 3 cannibals, boat on left bank, 0 on right
    start_state = State(3, 3, 1, 0, 0)
    if start_state.is_goal():
        return [start_state]

    queue = deque([[start_state]])
    visited = {start_state}

    while queue:
        path = queue.popleft()
        current_state = path[-1]

        if current_state.is_goal():
            return path

        for successor in get_successors(current_state):
            if successor not in visited:
                visited.add(successor)
                new_path = list(path)
                new_path.append(successor)
                queue.append(new_path)
    return None

print("Solving Missionaries and Cannibals Problem...")
solution = solve_missionaries_cannibals()

if solution:
    print(f"Solution found in {len(solution) - 1} steps:\n")
    for step, state in enumerate(solution):
        print(f"Step {step}: {state}")
else:
    print("No solution exists.")

Solving Missionaries and Cannibals Problem...
Solution found in 11 steps:

Step 0: (M_left: 3, C_left: 3, Boat: Left, M_right: 0, C_right: 0)
Step 1: (M_left: 3, C_left: 1, Boat: Right, M_right: 0, C_right: 2)
Step 2: (M_left: 3, C_left: 2, Boat: Left, M_right: 0, C_right: 1)
Step 3: (M_left: 3, C_left: 0, Boat: Right, M_right: 0, C_right: 3)
Step 4: (M_left: 3, C_left: 1, Boat: Left, M_right: 0, C_right: 2)
Step 5: (M_left: 1, C_left: 1, Boat: Right, M_right: 2, C_right: 2)
Step 6: (M_left: 2, C_left: 2, Boat: Left, M_right: 1, C_right: 1)
Step 7: (M_left: 0, C_left: 2, Boat: Right, M_right: 3, C_right: 1)
Step 8: (M_left: 0, C_left: 3, Boat: Left, M_right: 3, C_right: 0)
Step 9: (M_left: 0, C_left: 1, Boat: Right, M_right: 3, C_right: 2)
Step 10: (M_left: 1, C_left: 1, Boat: Left, M_right: 2, C_right: 2)
Step 11: (M_left: 0, C_left: 0, Boat: Right, M_right: 3, C_right: 3)


6.Write the python program for Vacuum Cleaner problem

In [ ]:
class VacuumCleanerEnvironment:
    def __init__(self):
        # 1 represents Dirty, 0 represents Clean
        # Location: 'A' or 'B'
        self.location = 'A'
        self.status = {'A': 1, 'B': 1}

    def run_simulation(self):
        print(f"Initial State: Location = {self.location}, Status = {self.status}\n")

        steps = 0
        while 1 in self.status.values() and steps < 4:
            current_loc = self.location
            print(f"--- Step {steps + 1} ---")
            print(f"Vacuum is currently in Room {current_loc}.")

            if self.status[current_loc] == 1:  # If Dirty
                print(f"Room {current_loc} is DIRTY. Action: SUCK.")
                self.status[current_loc] = 0   # Clean the room
                print(f"Room {current_loc} is now CLEAN. Current Status: {self.status}")
            else:
                print(f"Room {current_loc} is already CLEAN.")

            # If there's still dirty rooms, move to the other room
            if 1 in self.status.values():
                next_loc = 'B' if current_loc == 'A' else 'A'
                print(f"Action: Move to Room {next_loc}.")
                self.location = next_loc

            print()
            steps += 1

        print("Simulation Finished! All rooms are clean.")
        print(f"Final State: Location = {self.location}, Status = {self.status}")

# Instantiate and run the simulation
env = VacuumCleanerEnvironment()
env.run_simulation()

Initial State: Location = A, Status = {'A': 1, 'B': 1}

--- Step 1 ---
Vacuum is currently in Room A.
Room A is DIRTY. Action: SUCK.
Room A is now CLEAN. Current Status: {'A': 0, 'B': 1}
Action: Move to Room B.

--- Step 2 ---
Vacuum is currently in Room B.
Room B is DIRTY. Action: SUCK.
Room B is now CLEAN. Current Status: {'A': 0, 'B': 0}

Simulation Finished! All rooms are clean.
Final State: Location = B, Status = {'A': 0, 'B': 0}


7.Write the python program to implement BFS.

In [ ]:
from collections import deque

def bfs(graph, start_node):
    # Keep track of visited nodes to avoid processing the same node multiple times
    visited = set()
    # Use a queue to manage the order of node traversal
    queue = deque([start_node])

    # Mark the start node as visited
    visited.add(start_node)

    bfs_order = []

    while queue:
        # Dequeue a vertex from the queue
        current_node = queue.popleft()
        bfs_order.append(current_node)

        # Get all adjacent vertices of the dequeued vertex
        # If an adjacent vertex has not been visited, mark it visited and enqueue it
        for neighbor in graph.get(current_node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return bfs_order

# Example Graph represented as an adjacency list
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

start_node = 'A'
print(f"BFS starting from node '{start_node}':")
result = bfs(graph, start_node)
print(" -> ".join(result))

BFS starting from node 'A':
A -> B -> C -> D -> E -> F


8.Write the python program to implement DFS.

In [ ]:
def dfs_recursive(graph, node, visited=None, dfs_order=None):
    if visited is None:
        visited = set()
    if dfs_order is None:
        dfs_order = []

    # Mark the current node as visited and record it
    visited.add(node)
    dfs_order.append(node)

    # Recur for all adjacent vertices of this vertex
    for neighbor in graph.get(node, []):
        if neighbor not in visited:
            dfs_recursive(graph, neighbor, visited, dfs_order)

    return dfs_order

# Example Graph represented as an adjacency list
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

start_node = 'A'
print(f"DFS starting from node '{start_node}':")
result = dfs_recursive(graph, start_node)
print(" -> ".join(result))

DFS starting from node 'A':
A -> B -> D -> E -> F -> C


9.Write the python to implement Travelling Salesman Problem

In [ ]:
from itertools import permutations

def solve_tsp(graph, start):
    # Get all vertices except the starting vertex
    vertices = list(graph.keys())
    vertices.remove(start)

    min_path_weight = float('inf')
    best_path = []

    # Generate all possible permutations of vertices
    for perm in permutations(vertices):
        current_path_weight = 0
        current_node = start
        possible_path = [start]

        # Traverse the current permutation path
        possible = True
        for neighbor in perm:
            # If there is an edge between current_node and neighbor
            if neighbor in graph[current_node]:
                current_path_weight += graph[current_node][neighbor]
                current_node = neighbor
                possible_path.append(neighbor)
            else:
                possible = False
                break

        # Add the weight to return back to the starting node
        if possible and start in graph[current_node]:
            current_path_weight += graph[current_node][start]
            possible_path.append(start)

            # Keep track of the minimum weight and the best path
            if current_path_weight < min_path_weight:
                min_path_weight = current_path_weight
                best_path = possible_path

    return min_path_weight, best_path

# Representation of a complete graph with distances as weights
# keys are vertices, values are dicts of {neighbor: distance}
tsp_graph = {
    'A': {'A': 0, 'B': 10, 'C': 15, 'D': 20},
    'B': {'A': 10, 'B': 0, 'C': 35, 'D': 25},
    'C': {'A': 15, 'B': 35, 'C': 0, 'D': 30},
    'D': {'A': 20, 'B': 25, 'C': 30, 'D': 0}
}

start_city = 'A'
min_cost, path = solve_tsp(tsp_graph, start_city)

print(f"Solving TSP starting and ending at '{start_city}':")
print(f"Minimum Cost: {min_cost}")
print(f"Path: {' -> '.join(path)}")

Solving TSP starting and ending at 'A':
Minimum Cost: 80
Path: A -> B -> D -> C -> A


10.Write the python program to implement A* algorithm

In [ ]:
import heapq

def a_star_search(graph, start, goal, heuristics):
    # priority queue stores tuples of (f_score, current_node, path, g_score)
    # f_score = g_score + heuristic
    open_set = []
    heapq.heappush(open_set, (heuristics[start], start, [start], 0))

    visited = {}  # Keeps track of the minimum g_score to reach each node

    while open_set:
        f_score, current, path, g_score = heapq.heappop(open_set)

        # If goal is reached, return the path and total path cost
        if current == goal:
            return path, g_score

        # If we have already found a cheaper way to reach this node, skip it
        if current in visited and visited[current] <= g_score:
            continue
        visited[current] = g_score

        for neighbor, cost in graph.get(current, {}).items():
            new_g = g_score + cost
            # If we find a better path to the neighbor, explore it
            if neighbor not in visited or new_g < visited[neighbor]:
                new_f = new_g + heuristics.get(neighbor, 0)
                heapq.heappush(open_set, (new_f, neighbor, path + [neighbor], new_g))

    return None, float('inf')

# Example Graph (Weighted)
graph = {
    'A': {'B': 4, 'C': 3},
    'B': {'F': 5, 'E': 12},
    'C': {'D': 7, 'E': 10},
    'D': {'E': 2},
    'E': {'G': 5},
    'F': {'G': 16},
    'G': {}
}

# Straight-line distance heuristics to Goal 'G'
heuristics = {
    'A': 14,
    'B': 12,
    'C': 11,
    'D': 6,
    'E': 4,
    'F': 11,
    'G': 0
}

start_node = 'A'
goal_node = 'G'
path, total_cost = a_star_search(graph, start_node, goal_node, heuristics)

print(f"Solving A* Search from '{start_node}' to '{goal_node}':")
if path:
    print(f"Shortest Path: {' -> '.join(path)}")
    print(f"Total Path Cost: {total_cost}")
else:
    print("No path found.")

Solving A* Search from 'A' to 'G':
Shortest Path: A -> C -> D -> E -> G
Total Path Cost: 17


11. Write the python program for Map Coloring to implement CSP.

In [ ]:
def is_valid_assignment(node, color, assignment, neighbors):
    # Check if any assigned neighbor has the same color
    for neighbor in neighbors.get(node, []):
        if neighbor in assignment and assignment[neighbor] == color:
            return False
    return True

def backtrack_map_coloring(variables, domains, assignment, neighbors):
    # If all variables are assigned, we have a complete solution
    if len(assignment) == len(variables):
        return assignment

    # Select an unassigned variable
    unassigned = [v for v in variables if v not in assignment]
    node = unassigned[0]

    # Try assigning each available color in the domain
    for color in domains[node]:
        if is_valid_assignment(node, color, assignment, neighbors):
            assignment[node] = color

            # Recur to assign colors to remaining regions
            result = backtrack_map_coloring(variables, domains, assignment, neighbors)
            if result is not None:
                return result

            # If it leads to a dead end, backtrack
            del assignment[node]

    return None

# Define the map of Australia (variables and neighbors)
australia_regions = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T']

australia_neighbors = {
    'WA': ['NT', 'SA'],
    'NT': ['WA', 'SA', 'Q'],
    'SA': ['WA', 'NT', 'Q', 'NSW', 'V'],
    'Q': ['NT', 'SA', 'NSW'],
    'NSW': ['Q', 'SA', 'V'],
    'V': ['SA', 'NSW'],
    'T': []  # Tasmania is an island with no land neighbors
}

# Available colors (domains)
colors = ['Red', 'Green', 'Blue']
australia_domains = {region: colors for region in australia_regions}

print("Solving Map Coloring for Australia...")
solution = backtrack_map_coloring(australia_regions, australia_domains, {}, australia_neighbors)

if solution:
    print("\nSolution Found:")
    for region, color in sorted(solution.items()):
        print(f"{region}: {color}")
else:
    print("No valid coloring found with the given colors.")

Solving Map Coloring for Australia...

Solution Found:
NSW: Green
NT: Green
Q: Red
SA: Blue
T: Red
V: Red
WA: Red


12.Write the python program for Tic Tac Toe game

In [ ]:
def print_board(board):
    print("\n")
    print(f" {board[0]} | {board[1]} | {board[2]} ")
    print("---|---|---")
    print(f" {board[3]} | {board[4]} | {board[5]} ")
    print("---|---|---")
    print(f" {board[6]} | {board[7]} | {board[8]} ")
    print("\n")

def check_win(board):
    # Winning combinations
    win_conditions = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],  # Rows
        [0, 3, 6], [1, 4, 7], [2, 5, 8],  # Columns
        [0, 4, 8], [2, 4, 6]              # Diagonals
    ]
    for condition in win_conditions:
        if board[condition[0]] == board[condition[1]] == board[condition[2]] != ' ':
            return board[condition[0]]
    return None

def check_tie(board):
    return ' ' not in board

def play_game():
    board = [' '] * 9
    current_player = 'X'

    print("Welcome to Tic Tac Toe!")
    print("Positions are numbered from 1 to 9 as follows:")
    print(" 1 | 2 | 3 ")
    print("---|---|---")
    print(" 4 | 5 | 6 ")
    print("---|---|---")
    print(" 7 | 8 | 9 ")

    # Simulate a full game automatically as input is non-interactive in normal cell runs
    # Players make dummy predefined valid moves to demonstrate the game execution
    predefined_moves = [5, 1, 3, 7, 9, 2, 6, 4, 8]
    move_index = 0

    while True:
        print_board(board)
        winner = check_win(board)
        if winner:
            print(f"Player '{winner}' wins!")
            break
        if check_tie(board):
            print("It's a tie!")
            break

        if move_index < len(predefined_moves):
            choice = predefined_moves[move_index]
            move_index += 1
            print(f"Player '{current_player}' chooses position {choice}.")
        else:
            print("No more moves left to simulate.")
            break

        if board[choice - 1] == ' ':
            board[choice - 1] = current_player
            current_player = 'O' if current_player == 'X' else 'X'
        else:
            print("Position already taken! Skipping.")

# Run the simulated game
play_game()

Welcome to Tic Tac Toe!
Positions are numbered from 1 to 9 as follows:
 1 | 2 | 3 
---|---|---
 4 | 5 | 6 
---|---|---
 7 | 8 | 9 


   |   |   
---|---|---
   |   |   
---|---|---
   |   |   


Player 'X' chooses position 5.


   |   |   
---|---|---
   | X |   
---|---|---
   |   |   


Player 'O' chooses position 1.


 O |   |   
---|---|---
   | X |   
---|---|---
   |   |   


Player 'X' chooses position 3.


 O |   | X 
---|---|---
   | X |   
---|---|---
   |   |   


Player 'O' chooses position 7.


 O |   | X 
---|---|---
   | X |   
---|---|---
 O |   |   


Player 'X' chooses position 9.


 O |   | X 
---|---|---
   | X |   
---|---|---
 O |   | X 


Player 'O' chooses position 2.


 O | O | X 
---|---|---
   | X |   
---|---|---
 O |   | X 


Player 'X' chooses position 6.


 O | O | X 
---|---|---
   | X | X 
---|---|---
 O |   | X 


Player 'X' wins!


13.Write the python program to implement Minimax algorithm for gaming

In [ ]:
import math

def minimax(depth, node_index, is_maximizing, scores, target_depth):
    # Base case: target depth reached (terminal node with leaf values)
    if depth == target_depth:
        return scores[node_index]

    if is_maximizing:
        best = -math.inf
        # Recur for left and right children
        for i in range(2):
            val = minimax(depth + 1, node_index * 2 + i, False, scores, target_depth)
            best = max(best, val)
        return best
    else:
        best = math.inf
        # Recur for left and right children
        for i in range(2):
            val = minimax(depth + 1, node_index * 2 + i, True, scores, target_depth)
            best = min(best, val)
        return best

# Example of a game tree with 8 leaf values representing outcomes
# Level 0 (Root, Max's turn)
# Level 1 (Min's turn)
# Level 2 (Max's turn)
# Level 3 (Leaf nodes containing values)
leaf_scores = [3, 5, 2, 9, 12, 5, 23, 23]
tree_depth = int(math.log2(len(leaf_scores)))

print("Leaf node scores:", leaf_scores)
optimal_value = minimax(0, 0, True, leaf_scores, tree_depth)
print("The optimal value determined by Minimax for the Maximizing Player (root) is:", optimal_value)

Leaf node scores: [3, 5, 2, 9, 12, 5, 23, 23]
The optimal value determined by Minimax for the Maximizing Player (root) is: 12


14.Write the python program to implement Apha & Beta pruning algorithm for gaming


In [ ]:
import math

def alpha_beta_pruning(depth, node_index, is_maximizing, alpha, beta, scores, target_depth):
    # Base case: target depth reached (terminal node with leaf values)
    if depth == target_depth:
        return scores[node_index]

    if is_maximizing:
        best = -math.inf
        # Recur for left and right children
        for i in range(2):
            val = alpha_beta_pruning(depth + 1, node_index * 2 + i, False, alpha, beta, scores, target_depth)
            best = max(best, val)
            alpha = max(alpha, best)

            # Alpha-Beta Pruning condition
            if beta <= alpha:
                print(f"Pruned at Maximizer Node: index {node_index * 2 + i} on depth {depth + 1} (Alpha: {alpha}, Beta: {beta})")
                break
        return best
    else:
        best = math.inf
        # Recur for left and right children
        for i in range(2):
            val = alpha_beta_pruning(depth + 1, node_index * 2 + i, True, alpha, beta, scores, target_depth)
            best = min(best, val)
            beta = min(beta, best)

            # Alpha-Beta Pruning condition
            if beta <= alpha:
                print(f"Pruned at Minimizer Node: index {node_index * 2 + i} on depth {depth + 1} (Alpha: {alpha}, Beta: {beta})")
                break
        return best

# Example of a game tree with 8 leaf values representing outcomes
# Level 0 (Root, Max's turn)
# Level 1 (Min's turn)
# Level 2 (Max's turn)
# Level 3 (Leaf nodes containing values)
leaf_scores = [3, 5, 6, 9, 1, 2, 0, -1]
tree_depth = int(math.log2(len(leaf_scores)))

print("Leaf node scores:", leaf_scores)
optimal_value = alpha_beta_pruning(0, 0, True, -math.inf, math.inf, leaf_scores, tree_depth)
print("The optimal value determined by Alpha-Beta Pruning is:", optimal_value)

Leaf node scores: [3, 5, 6, 9, 1, 2, 0, -1]
Pruned at Maximizer Node: index 2 on depth 3 (Alpha: 6, Beta: 5)
Pruned at Minimizer Node: index 2 on depth 2 (Alpha: 5, Beta: 2)
The optimal value determined by Alpha-Beta Pruning is: 5


15.Write the python program to implement Decision Tree

In [ ]:
import numpy as np

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None

class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=100, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Check terminal conditions
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)

        # Find the best split
        best_feature, best_thresh = self._best_split(X, y, feat_idxs)

        # Create child nodes
        left_idxs, right_idxs = self._split(X[:, best_feature], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(feature=best_feature, threshold=best_thresh, left=left, right=right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thresh in thresholds:
                gain = self._information_gain(y, X_column, thresh)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh

        return split_idx, split_thresh

    def _information_gain(self, y, X_column, threshold):
        # Parent entropy
        parent_entropy = self._entropy(y)

        # Create children
        left_idxs, right_idxs = self._split(X_column, threshold)

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        # Weighted average of children entropy
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        # Information Gain is the reduction in entropy
        information_gain = parent_entropy - child_entropy
        return information_gain

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _most_common_label(self, y):
        return np.bincount(y).argmax()

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

# Example Usage
if __name__ == "__main__":
    # Toy Dataset:
    # Features: [Is_Rainy, Has_Wind]
    # Target: Go_Out_To_Play (0 = No, 1 = Yes)
    X_train = np.array([
        [0, 0],  # Sunny, Calm
        [0, 1],  # Sunny, Windy
        [1, 0],  # Rainy, Calm
        [1, 1],  # Rainy, Windy
        [0, 0],  # Sunny, Calm
        [1, 0],  # Rainy, Calm
    ])
    y_train = np.array([1, 1, 0, 0, 1, 0])

    clf = DecisionTree(max_depth=3)
    clf.fit(X_train, y_train)

    # Test on a new observation: [Sunny, Windy]
    test_sample = np.array([[0, 1]])
    prediction = clf.predict(test_sample)

    print("--- Decision Tree Classifier ---")
    print("Training Features (Rainy, Windy):\n", X_train)
    print("Training Labels (Play):", y_train)
    print(f"Prediction for sample {test_sample[0]} (0=No Play, 1=Play):", prediction[0])

--- Decision Tree Classifier ---
Training Features (Rainy, Windy):
 [[0 0]
 [0 1]
 [1 0]
 [1 1]
 [0 0]
 [1 0]]
Training Labels (Play): [1 1 0 0 1 0]
Prediction for sample [0 1] (0=No Play, 1=Play): 1


16.Write the python program to implement Feed forward neural Network

In [ ]:
import numpy as np

# Sigmoid activation function and its derivative
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

class FeedForwardNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.lr = learning_rate
        # Initialize weights randomly with mean 0
        self.weights_input_hidden = np.random.uniform(-1, 1, (input_size, hidden_size))
        self.weights_hidden_output = np.random.uniform(-1, 1, (hidden_size, output_size))
        # Initialize biases
        self.bias_hidden = np.zeros((1, hidden_size))
        self.bias_output = np.zeros((1, output_size))

    def forward(self, X):
        # Forward pass through the network
        self.hidden_input = np.dot(X, self.weights_input_hidden) + self.bias_hidden
        self.hidden_output = sigmoid(self.hidden_input)

        self.final_input = np.dot(self.hidden_output, self.weights_hidden_output) + self.bias_output
        self.final_output = sigmoid(self.final_input)
        return self.final_output

    def backward(self, X, y, output):
        # Backpropagation of errors
        error_output = y - output
        delta_output = error_output * sigmoid_derivative(output)

        error_hidden = delta_output.dot(self.weights_hidden_output.T)
        delta_hidden = error_hidden * sigmoid_derivative(self.hidden_output)

        # Update weights and biases
        self.weights_hidden_output += self.hidden_output.T.dot(delta_output) * self.lr
        self.weights_input_hidden += X.T.dot(delta_hidden) * self.lr
        self.bias_output += np.sum(delta_output, axis=0, keepdims=True) * self.lr
        self.bias_hidden += np.sum(delta_hidden, axis=0, keepdims=True) * self.lr

    def train(self, X, y, epochs=10000):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output)
            if epoch % 2000 == 0:
                loss = np.mean(np.square(y - output))
                print(f"Epoch {epoch:5d} | Mean Squared Error: {loss:.6f}")

# Training data for XOR logical gate classification problem
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y_xor = np.array([
    [0],
    [1],
    [1],
    [0]
])

# Initialize and train the network
nn = FeedForwardNeuralNetwork(input_size=2, hidden_size=4, output_size=1, learning_rate=0.2)
print("--- Training Feedforward Neural Network (XOR problem) ---")
nn.train(X_xor, y_xor, epochs=10001)

# Evaluate predictions
predictions = nn.forward(X_xor)
print("\n--- Final Model Predictions ---")
for i in range(len(X_xor)):
    print(f"Input: {X_xor[i]} | Predicted: {predictions[i][0]:.4f} | Target: {y_xor[i][0]}")

--- Training Feedforward Neural Network (XOR problem) ---
Epoch     0 | Mean Squared Error: 0.259171
Epoch  2000 | Mean Squared Error: 0.025224
Epoch  4000 | Mean Squared Error: 0.003696
Epoch  6000 | Mean Squared Error: 0.001803
Epoch  8000 | Mean Squared Error: 0.001165
Epoch 10000 | Mean Squared Error: 0.000851

--- Final Model Predictions ---
Input: [0 0] | Predicted: 0.0166 | Target: 0
Input: [0 1] | Predicted: 0.9728 | Target: 1
Input: [1 0] | Predicted: 0.9722 | Target: 1
Input: [1 1] | Predicted: 0.0402 | Target: 0
